# Communication-Aware V2V Perception — Approach Pipeline Testing

This notebook is a clean testing notebook for the full communication-aware V2V pipeline.

It uses **approach-based names**, not development/phase names. The workflow is:

1. Verify environment, repo, data, and checkpoints.
2. Run local compile/tests.
3. Run small smoke tests for implemented communication approaches.
4. Optionally run controlled full inference after smoke tests pass.
5. Collect summaries, debug maps, and backup outputs.

Smoke tests validate pipeline correctness and metric generation. They are **not final AP results**.

## 2. Environment and Paths

Adjust these paths only if your Kaggle input names change.

In [ ]:
from pathlib import Path
import os
import json
import shutil
import subprocess
import textwrap

# cumm/spconv may incorrectly try to JIT-build helper bindings on Kaggle.
# Disable that path globally before any spconv/cumm import.
os.environ.setdefault('CUMM_DISABLE_JIT', '1')

REPO_DIR = Path('/kaggle/working/comm-aware-v2v-perception')
PY = Path('/kaggle/working/v2v_env/bin/python')
MODEL_DIR = Path('/kaggle/input/best-epoch')
DATA_ROOT = Path('/kaggle/input/data-all')
RUNS_ROOT = Path('/kaggle/working/approach_runs')

CARLA_TEST_DIR = DATA_ROOT / 'test/test'
CULVER_TEST_DIR = DATA_ROOT / 'test/test_culver_city/test_culver_city'

RUNS_ROOT.mkdir(parents=True, exist_ok=True)

print('REPO_DIR:', REPO_DIR, 'exists=', REPO_DIR.exists())
print('PY:', PY, 'exists=', PY.exists())
print('MODEL_DIR:', MODEL_DIR, 'exists=', MODEL_DIR.exists())
print('DATA_ROOT:', DATA_ROOT, 'exists=', DATA_ROOT.exists())
print('CARLA_TEST_DIR:', CARLA_TEST_DIR, 'exists=', CARLA_TEST_DIR.exists())
print('CULVER_TEST_DIR:', CULVER_TEST_DIR, 'exists=', CULVER_TEST_DIR.exists())
print('RUNS_ROOT:', RUNS_ROOT)


## 2.1 Install / Verify Required Packages

Run this before compile tests and smoke inference. Kaggle sometimes has many packages preinstalled, but inference imports `open3d`, `torch`, `spconv`, `yaml`, and other dependencies. This cell installs only missing basics and then verifies the runtime.

In [ ]:
import sys
import subprocess
from pathlib import Path

# If v2v_env does not exist, use the current notebook Python.
if not PY.exists():
    print('v2v_env not found. Using current notebook Python instead.')
    PY = Path(sys.executable)
print('Active PY:', PY)

def py_has_module(module_name):
    cmd = f'{PY} -c "import {module_name}"'
    proc = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    return proc.returncode == 0

required = {
    'yaml': 'pyyaml',
    'open3d': 'open3d',
    'tqdm': 'tqdm',
    'easydict': 'easydict',
    'timm': 'timm',
    'pandas': 'pandas',
    'numpy': 'numpy',
}
packages = [pkg for module, pkg in required.items() if not py_has_module(module)]

if packages:
    cmd = [str(PY), '-m', 'pip', 'install', '-q'] + packages
    print('Installing:', ' '.join(packages))
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout[-8000:])
    print('exit:', proc.returncode)
    if proc.returncode != 0:
        raise RuntimeError('Package installation failed')
else:
    print('Basic Python packages already available in active PY.')

# Import notebook-side packages after installation.
import pandas as pd
import yaml
import numpy as np


In [ ]:
print('CUMM_DISABLE_JIT:', os.environ.get('CUMM_DISABLE_JIT'))
# Verify runtime imports needed by smoke inference.
# This cell reports status only; if spconv fails, run the repair cell below.
verify_imports = [
    "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())",
    "import yaml; print('yaml ok')",
    "import open3d; print('open3d ok')",
    "import tqdm; print('tqdm ok')",
    "import easydict; print('easydict ok')",
    "import timm; print('timm ok')",
    "import spconv; from spconv.utils import Point2VoxelCPU3d; print('spconv ok')",
]

runtime_import_status = {}
for snippet in verify_imports:
    cmd = f'{PY} -c "{snippet}"'
    print('\n$ ' + cmd)
    proc = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout[-4000:])
    print('exit:', proc.returncode)
    runtime_import_status[snippet] = proc.returncode


### 2.2 Optional: Repair spconv Install

Run this only if `import spconv` fails. For Kaggle Python 3.12, use the `cu121` binary wheel pair. The repository now has a safe fallback for `cumm.tensorview`, so we keep the `cumm` version compatible with `spconv-cu121==2.3.8`.


In [ ]:
# Run only if: import spconv fails.
# This installs binary wheels only. It avoids broken source/JIT builds.
import sys
os.environ.setdefault('CUMM_DISABLE_JIT', '1')

if not PY.exists():
    PY = Path(sys.executable)

py_tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
if py_tag == 'cp312':
    # spconv-cu121 2.3.8 requires cumm-cu121 >=0.7.11,<0.8.0.
    spconv_pkgs = ['cumm-cu121==0.7.11', 'spconv-cu121==2.3.8']
else:
    # Python 3.10/3.11 Kaggle-like environments can also use cu120.
    spconv_pkgs = ['cumm-cu120==0.6.3', 'spconv-cu120==2.3.6']

repair_cmds = [
    [str(PY), '-m', 'pip', 'uninstall', '-y', 'spconv', 'cumm', 'spconv-cu120', 'cumm-cu120', 'spconv-cu121', 'cumm-cu121'],
    [str(PY), '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel', 'ninja', 'pccm', 'ccimport', 'pybind11'],
    [str(PY), '-m', 'pip', 'install', '-q', '--no-cache-dir', '--only-binary=:all:'] + spconv_pkgs,
]

print('Python tag:', py_tag)
print('Selected spconv packages:', spconv_pkgs)
for cmd in repair_cmds:
    print('\n$ ' + ' '.join(cmd))
    proc = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout[-12000:])
    print('exit:', proc.returncode)
    if proc.returncode != 0:
        raise RuntimeError('spconv repair command failed')

verify_snippets = [
    "import spconv; print('spconv import ok')",
    "from spconv.utils import Point2VoxelCPU3d; print('Point2VoxelCPU3d ok')",
    "import importlib; importlib.import_module('cumm.tensorview'); print('cumm.tensorview ok')",
]
for snippet in verify_snippets:
    cmd = f'{PY} -c "{snippet}"'
    print('\n$ ' + cmd)
    proc = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout[-12000:])
    print('exit:', proc.returncode)
    if proc.returncode != 0:
        raise RuntimeError('spconv verification failed')


In [ ]:
# Fallback diagnostic cell. Run only if the repair cell above fails.
# In Python 3.12, cu120 wheels for spconv are not available, so cu121 is the correct target.
print('Python:', sys.version)
print('PY:', PY)
for pkg in ['spconv', 'spconv-cu120', 'cumm-cu120', 'spconv-cu121', 'cumm-cu121', 'cumm']:
    cmd = f'{PY} -m pip show {pkg}'
    print('\n$ ' + cmd)
    proc = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout[-3000:])


In [ ]:
# Clone or update repo. Safe for a fresh Kaggle run from the first cell.
# If the repo already exists, pull latest main so notebook and code agree.
repo_url = 'https://github.com/mohsenshahverdy/comm-aware-v2v-perception.git'
if not REPO_DIR.exists():
    cmd = f'cd /kaggle/working && git clone {repo_url}'
    print(cmd)
    subprocess.run(cmd, shell=True, check=True)
elif (REPO_DIR / '.git').exists():
    print('Repo already exists; updating:', REPO_DIR)
    subprocess.run('git fetch origin main', cwd=str(REPO_DIR), shell=True, check=True)
    subprocess.run('git reset --hard origin/main', cwd=str(REPO_DIR), shell=True, check=True)
else:
    raise RuntimeError(f'REPO_DIR exists but is not a git repo: {REPO_DIR}')

print('Repo ready:', REPO_DIR)


## 3. Git / Repo Verification

These checks confirm that the notebook is using the intended repository state.

In [ ]:
def run_cmd(cmd, cwd=REPO_DIR, check=False):
    print('\n$ ' + cmd)
    proc = subprocess.run(cmd, cwd=str(cwd), shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout)
    if check and proc.returncode != 0:
        raise RuntimeError(f'Command failed: {cmd}')
    return proc

run_cmd('git branch --show-current')
run_cmd('git rev-parse --short HEAD')
run_cmd('git log --oneline -5')
run_cmd('git log --oneline --all --grep="V2VAM" -10 || true')
run_cmd('grep -En "temporal_receiver_request_energy_topk_10|receiver_request_energy_topk_10|selective_topk_energy_10" src/hypes_yaml/communication_approach_presets.yaml')

## 3.1 Build Required Native Extensions

Inference imports compiled native modules such as `src.utils.box_overlaps`. Build these before smoke tests. If a module already exists, rebuilding is harmless.

In [ ]:
# Required for: from src.utils.box_overlaps import bbox_overlaps
run_test_cmds = [
    f'cd {REPO_DIR} && {PY} src/utils/setup.py build_ext --inplace',
    f'cd {REPO_DIR} && {PY} src/pcdet_utils/setup.py build_ext --inplace',
]

for cmd in run_test_cmds:
    print('\n$ ' + cmd)
    proc = subprocess.run(cmd, shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout[-8000:])
    print('exit:', proc.returncode)
    if proc.returncode != 0:
        print('Build failed. Fix this before running inference smoke tests.')

In [ ]:
# Verify compiled extension imports.
verify_cmd = f'PYTHONPATH={REPO_DIR} {PY} -c "from src.utils.box_overlaps import bbox_overlaps; print(\'box_overlaps ok\')"'
print('$ ' + verify_cmd)
proc = subprocess.run(verify_cmd, cwd=str(REPO_DIR), shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(proc.stdout)
print('exit:', proc.returncode)

## 4. Import and Tool Checks

Run these lightweight checks before any smoke test.

In [ ]:
def run_test(name, cmd):
    print(f'\n===== {name} =====')
    proc = subprocess.run(cmd, cwd=str(REPO_DIR), shell=True, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout[-5000:])
    if proc.returncode == 0:
        print(f'PASS: {name}')
    else:
        print(f'FAIL: {name} exit={proc.returncode}')
    return proc.returncode

compile_cmd = (
    f'PYTHONPATH={REPO_DIR} {PY} -m py_compile '
    'src/tools/testing/smoke_test_pipeline.py '
    'src/tools/inference.py '
    'src/models/fuse_modules/communication_policy.py '
    'src/models/fuse_modules/V2VAM.py'
)
run_test('py_compile core files', compile_cmd)

In [ ]:
LOCAL_TESTS = [
    'src.tools.testing.test_comm_policy_fake',
    'src.tools.testing.test_v2vam_correctness',
    'src.tools.testing.test_temporal_cache',
    'src.tools.testing.test_temporal_metadata',
    'src.tools.testing.test_temporal_receiver_request',
    'src.tools.testing.test_centralized_logger',
]

local_test_results = {}
for mod in LOCAL_TESTS:
    cmd = f'PYTHONPATH={REPO_DIR} {PY} -m {mod}'
    local_test_results[mod] = run_test(mod, cmd)

print('\nSummary:')
for mod, code_ in local_test_results.items():
    print(('PASS' if code_ == 0 else 'FAIL'), mod)

## 5. Approach Registry

Only approaches in `RUNNABLE_APPROACHES` should be executed. Planned approaches are documented for roadmap clarity but are not runnable here.

In [ ]:
RUNNABLE_APPROACHES = {
    'baseline_full_communication': {
        'family': 'baseline',
        'description': 'Baseline config with communication disabled/full local accounting baseline.',
        'recommended_split': 'carla',
        'debug_maps_useful': False,
        'expected_special_metrics': ['comm_total_normalized_ratio'],
    },
    'measurement_full_communication': {
        'family': 'measurement',
        'description': 'Full communication measurement baseline.',
        'recommended_split': 'carla',
        'debug_maps_useful': False,
        'expected_special_metrics': ['comm_total_bytes_per_frame'],
    },
    'selective_topk_energy_10': {
        'family': 'selective',
        'description': 'Sender-side/top-k energy selection at 10% collaborator cells.',
        'recommended_split': 'carla',
        'debug_maps_useful': False,
        'expected_special_metrics': ['comm_active_ratio'],
    },
    'selective_random_comm_only_10': {
        'family': 'selective',
        'description': 'Random collaborator-only spatial masking at 10%. Ego remains local/unchanged.',
        'recommended_split': 'carla',
        'debug_maps_useful': False,
        'expected_special_metrics': ['comm_active_ratio'],
    },
    'receiver_request_energy_topk_10': {
        'family': 'receiver_request',
        'description': 'Receiver-driven request based on inverse ego energy and collaborator L2 context.',
        'recommended_split': 'carla',
        'debug_maps_useful': True,
        'expected_special_metrics': ['receiver_request_keep_ratio', 'comm_context_normalized_ratio'],
    },
    'temporal_receiver_request_energy_topk_10': {
        'family': 'temporal_receiver_request',
        'description': 'Temporal receiver-driven request with cache, novelty, age, and confidence factors.',
        'recommended_split': 'carla',
        'debug_maps_useful': True,
        'expected_special_metrics': ['temporal_novelty_mean', 'temporal_cache_hit_ratio', 'comm_total_normalized_ratio_after_init'],
    },
}

OPTIONAL_SWEEPS = {
    'selective_topk_energy': ['selective_topk_energy_05', 'selective_topk_energy_10', 'selective_topk_energy_25', 'selective_topk_energy_50'],
    'receiver_request_energy_topk': ['receiver_request_energy_topk_05', 'receiver_request_energy_topk_10', 'receiver_request_energy_topk_25', 'receiver_request_energy_topk_50'],
    'temporal_receiver_request': ['temporal_receiver_request_energy_topk_10'],
}

PLANNED_APPROACHES = {
    'receiver_request_uncertainty_topk_10': 'planned: requires ego-only uncertainty/objectness before fusion',
    'receiver_request_visibility_topk': 'planned: requires BEV visibility/density/occlusion maps',
    'receiver_request_learned': 'planned: trainable request head',
    'receiver_request_learned_budget': 'planned: trainable request head with budget-aware loss',
    'receiver_request_warped': 'planned: BEV warp context/mask alignment',
    'temporal_receiver_request_uncertainty_topk_10': 'planned: temporal cache plus uncertainty need map',
    'temporal_receiver_request_visibility_topk_10': 'planned: temporal cache plus visibility/occlusion need map',
    'temporal_receiver_request_learned': 'planned: learned temporal request policy',
}

pd.DataFrame.from_dict(RUNNABLE_APPROACHES, orient='index')

## 6. Generic Smoke Test Function

This function runs `src.tools.testing.smoke_test_pipeline` with approach-based run names.

In [ ]:
def smoke_run_dir(approach, split='carla'):
    return RUNS_ROOT / f'smoke_{split}_{approach}'


def _load_json(path):
    with open(path, 'r') as f:
        return json.load(f)


def print_compact_report(report):
    print('\nSmoke status:', report.get('status'))
    print('Approach:', report.get('approach'))
    print('Split:', report.get('split'))
    print('Processed frames:', report.get('processed_frames'))
    metrics = report.get('metrics', {}) or {}
    keys = [
        'comm_total_normalized_ratio',
        'comm_feature_normalized_ratio',
        'comm_context_normalized_ratio',
        'comm_metadata_normalized_ratio',
        'comm_total_bytes_per_frame',
        'receiver_request_keep_ratio',
        'temporal_novelty_mean',
        'temporal_cache_hit_ratio',
        'temporal_init_frame_ratio',
        'comm_average_bytes_per_frame',
    ]
    for k in keys:
        if k in metrics and metrics[k] is not None:
            print(f'{k}: {metrics[k]}')
    if report.get('error_message'):
        print('Error:', report.get('error_message'))


def run_smoke(approach, split='carla', max_samples=20, skip_ap=True, save_debug_maps=False, force_clean=True):
    if approach not in RUNNABLE_APPROACHES:
        raise ValueError(f'Approach is not registered as runnable: {approach}')

    cmd = [
        str(PY), '-m', 'src.tools.testing.smoke_test_pipeline',
        '--approach', approach,
        '--checkpoint_dir', str(MODEL_DIR),
        '--split', split,
        '--max_samples', str(max_samples),
        '--runs_root', str(RUNS_ROOT),
    ]
    if skip_ap:
        cmd.append('--skip_ap')
    if save_debug_maps:
        cmd.append('--save_debug_maps')
    if force_clean:
        cmd.append('--force_clean')

    env = os.environ.copy()
    env['PYTHONPATH'] = str(REPO_DIR)
env['CUMM_DISABLE_JIT'] = '1'
    print('Command:')
    print(' '.join(cmd))
    proc = subprocess.run(cmd, cwd=str(REPO_DIR), env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(proc.stdout[-8000:])

    report_path = smoke_run_dir(approach, split) / 'smoke_test_report.json'
    if not report_path.exists():
        print('FAIL: smoke_test_report.json not found:', report_path)
        return {'status': 'fail', 'error_message': 'missing smoke_test_report.json', 'stdout': proc.stdout}

    report = _load_json(report_path)
    print_compact_report(report)

    if report.get('status') != 'pass':
        log_path = smoke_run_dir(approach, split) / 'smoke_inference.log'
        print('\nInference log tail:')
        print('Log path:', log_path)
        if log_path.exists():
            text = log_path.read_text(errors='replace')
            print(text[-12000:])
        else:
            print('Missing inference log:', log_path)

    return report

## 7. Single Smoke Tests

Start here. These are small and safe compared with full inference.

In [ ]:
# A. Smoke test selective baseline
report_selective = run_smoke('selective_topk_energy_10', max_samples=20)

In [ ]:
# B. Smoke test receiver request with debug maps
report_receiver = run_smoke('receiver_request_energy_topk_10', max_samples=20, save_debug_maps=True)

In [ ]:
# C. Smoke test temporal receiver request with debug maps
report_temporal = run_smoke('temporal_receiver_request_energy_topk_10', max_samples=20, save_debug_maps=True)

## 8. All Implemented Approaches Smoke Test

This runs all implemented approaches with only 5 samples each. Planned placeholders are skipped by default.

In [ ]:
cmd = [
    str(PY), '-m', 'src.tools.testing.smoke_test_pipeline',
    '--all_approaches',
    '--split', 'carla',
    '--max_samples', '5',
    '--checkpoint_dir', str(MODEL_DIR),
    '--runs_root', str(RUNS_ROOT),
    '--skip_ap',
]

env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR)
env['CUMM_DISABLE_JIT'] = '1'
print(' '.join(cmd))
proc = subprocess.run(cmd, cwd=str(REPO_DIR), env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(proc.stdout[-10000:])

index_csv = RUNS_ROOT / 'smoke_test_index.csv'
if index_csv.exists():
    smoke_index_df = pd.read_csv(index_csv)
    display(smoke_index_df)
else:
    print('Missing:', index_csv)

## 9. Debug Map Inspection

Use this section after running receiver-request or temporal smoke tests with `save_debug_maps=True`.

In [ ]:
def list_debug_maps():
    rr_maps = sorted(RUNS_ROOT.glob('**/receiver_request_debug/*.npz'))
    temporal_maps = sorted(RUNS_ROOT.glob('**/temporal_receiver_request_debug/*.npz'))
    print('Receiver-request debug maps:', len(rr_maps))
    for p in rr_maps[:20]:
        print(p)
    print('\nTemporal receiver-request debug maps:', len(temporal_maps))
    for p in temporal_maps[:20]:
        print(p)
    return rr_maps, temporal_maps

rr_maps, temporal_maps = list_debug_maps()

In [ ]:
def inspect_npz(path):
    path = Path(path)
    data = np.load(path)
    print('File:', path)
    for key in data.files:
        arr = data[key]
        print(f'{key:32s} shape={arr.shape} min={arr.min():.6f} max={arr.max():.6f} mean={arr.mean():.6f}')
    return data

# Example usage after debug maps exist:
# inspect_npz(temporal_maps[0])
# inspect_npz(rr_maps[0])

Expected temporal map keys:

- `ego_need_map`
- `collaborator_context_map`
- `previous_cache_map`
- `novelty_map`
- `temporal_factor_map`
- `request_score_map`
- `request_mask`
- `cache_age_map`
- `cache_confidence_map`

Expected receiver-request map keys:

- `ego_need_map`
- `collaborator_context_map`
- `request_score_map`
- `request_mask`

## 10. Optional Full Inference Runner

Run this only after smoke tests pass. This can consume GPU quota.

This function prepares a run folder and calls `src.tools.inference` without `--max_samples` and without `--skip_ap`.

In [ ]:
def prepare_full_run(approach, split='carla', save_debug_maps=False, force_clean=False):
    if approach not in RUNNABLE_APPROACHES:
        raise ValueError(f'Approach is not registered as runnable: {approach}')

    run_dir = RUNS_ROOT / f'full_{split}_{approach}'
    if run_dir.exists() and force_clean:
        shutil.rmtree(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    for p in MODEL_DIR.glob('net_epoch*.pth'):
        shutil.copy2(p, run_dir / p.name)
    if (MODEL_DIR / 'latest.pth').exists():
        shutil.copy2(MODEL_DIR / 'latest.pth', run_dir / 'latest.pth')

    src_cfg = REPO_DIR / 'src/hypes_yaml/point_pillar_intermediate_V2VAM.yaml'
    cfg = yaml.safe_load(open(src_cfg))
    cfg['communication_preset'] = approach
    cfg['root_dir'] = str(DATA_ROOT / 'train')
    cfg['validate_dir'] = str(CULVER_TEST_DIR if split == 'culver' else CARLA_TEST_DIR)

    if save_debug_maps:
        comm = cfg.setdefault('model', {}).setdefault('args', {}).setdefault('communication', {})
        rr = comm.setdefault('receiver_request', {})
        rr['save_request_maps'] = True
        rr['debug_num_frames'] = 5
        temporal = rr.setdefault('temporal', {})
        temporal['save_temporal_maps'] = True
        temporal['debug_num_frames'] = 5

    with open(run_dir / 'config.yaml', 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

    preset_src = REPO_DIR / 'src/hypes_yaml/communication_approach_presets.yaml'
    shutil.copy2(preset_src, run_dir / 'communication_approach_presets.yaml')
    return run_dir


def run_full_inference(approach, split='carla', save_debug_maps=False, force_clean=False):
    run_dir = prepare_full_run(approach, split=split, save_debug_maps=save_debug_maps, force_clean=force_clean)
    cmd = [
        str(PY), '-u', '-m', 'src.tools.inference',
        '--model_dir', str(run_dir),
        '--fusion_method', 'intermediate',
        '--global_sort_detections',
    ]
    env = os.environ.copy()
    env['PYTHONPATH'] = str(REPO_DIR)
env['CUMM_DISABLE_JIT'] = '1'
    print(' '.join(cmd))
    log_path = run_dir / 'full_inference.log'
    with open(log_path, 'w') as lf:
        proc = subprocess.run(cmd, cwd=str(REPO_DIR), env=env, stdout=lf, stderr=subprocess.STDOUT)
    print('Exit code:', proc.returncode)
    print('Log:', log_path)
    return run_dir, proc.returncode

# Do not execute by default.
# run_dir, code = run_full_inference('temporal_receiver_request_energy_topk_10', save_debug_maps=True)

## 11. Recommended Full Run Order

Run full inference only after smoke tests pass.

1. `selective_topk_energy_10`
2. `receiver_request_energy_topk_10`
3. `temporal_receiver_request_energy_topk_10`
4. Optional sweeps if the first comparison is promising.

## 12. Result Loading and Comparison

Collect summaries from smoke and full runs into a single dataframe.

In [ ]:
def load_summary(run_dir):
    run_dir = Path(run_dir)
    summary_path = run_dir / 'summary_eval.yaml'
    smoke_path = run_dir / 'smoke_test_report.json'
    data = {}
    if summary_path.exists():
        with open(summary_path, 'r') as f:
            data.update(yaml.safe_load(f) or {})
    if smoke_path.exists():
        smoke = _load_json(smoke_path)
        data['smoke_status'] = smoke.get('status')
        data['processed_frames'] = smoke.get('processed_frames')
        for k, v in (smoke.get('metrics') or {}).items():
            data.setdefault(k, v)
    return data


def collect_summaries(runs_root=RUNS_ROOT):
    rows = []
    for run_dir in sorted(Path(runs_root).iterdir()):
        if not run_dir.is_dir():
            continue
        data = load_summary(run_dir)
        if not data:
            continue
        name = run_dir.name
        approach = name
        for prefix in ['smoke_carla_', 'smoke_culver_', 'full_carla_', 'full_culver_']:
            if approach.startswith(prefix):
                approach = approach[len(prefix):]
        row = {
            'run_dir': str(run_dir),
            'approach': approach,
            'split': 'culver' if 'culver' in name else 'carla',
            'AP@0.3': data.get('ap30', data.get('ap_30')),
            'AP@0.5': data.get('ap_50'),
            'AP@0.7': data.get('ap_70'),
            'comm_feature_normalized_ratio': data.get('comm_feature_normalized_ratio'),
            'comm_context_normalized_ratio': data.get('comm_context_normalized_ratio'),
            'comm_metadata_normalized_ratio': data.get('comm_metadata_normalized_ratio'),
            'comm_total_normalized_ratio': data.get('comm_total_normalized_ratio', data.get('comm_normalized_ratio')),
            'comm_total_bytes_per_frame': data.get('comm_total_bytes_per_frame'),
            'receiver_request_keep_ratio': data.get('receiver_request_keep_ratio'),
            'temporal_novelty_mean': data.get('temporal_novelty_mean'),
            'temporal_cache_hit_ratio': data.get('temporal_cache_hit_ratio'),
            'temporal_init_frame_ratio': data.get('temporal_init_frame_ratio'),
            'comm_total_bytes_per_frame_after_init': data.get('comm_total_bytes_per_frame_after_init'),
            'comm_total_normalized_ratio_after_init': data.get('comm_total_normalized_ratio_after_init'),
            'smoke_status': data.get('smoke_status'),
            'processed_frames': data.get('processed_frames'),
        }
        rows.append(row)
    return pd.DataFrame(rows)

comparison_df = collect_summaries()
display(comparison_df)

## 13. Backup / Download

This creates a zip with approach run outputs and summary files. Use it before stopping the Kaggle session.

In [ ]:
BACKUP_ZIP = Path('/kaggle/working/approach_pipeline_test_outputs.zip')

items_to_backup = []
if RUNS_ROOT.exists():
    items_to_backup.append(str(RUNS_ROOT))
notebook_candidates = [
    Path('/kaggle/working/communication_approach_pipeline_testing.ipynb'),
    REPO_DIR / 'docs/notebooks/communication_approach_pipeline_testing.ipynb',
]
for p in notebook_candidates:
    if p.exists():
        items_to_backup.append(str(p))

cmd = ['zip', '-r', str(BACKUP_ZIP)] + items_to_backup
print(' '.join(cmd))
proc = subprocess.run(cmd, cwd='/kaggle/working', text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(proc.stdout[-5000:])
print('Backup:', BACKUP_ZIP, 'exists=', BACKUP_ZIP.exists())

## 14. Notes for Interpretation

- Smoke AP is not reliable.
- Smoke tests check pipeline correctness and metric/debug output.
- Full runs are required for reportable AP.
- For temporal approaches, inspect after-init metrics and cumulative bytes, not only average per-frame bytes.
- Debug maps should be checked before trusting temporal results.
- `comm_total_normalized_ratio` includes feature + context + metadata communication.
- `comm_total_normalized_ratio_after_init` is especially useful for temporal approaches because first-frame initialization can be expensive.

## 15. Safety Features

- Existing full runs are not deleted by default.
- `force_clean=True` is used only by smoke test helper calls.
- Run folders use approach-based names.
- Planned approaches are listed but not runnable.
- No model logic or communication policy behavior is modified by this notebook.